# Work Session 1: Document Preprocessing, Representation and Indexing

## Part 1A: Corpus Inspection
- **Uppercase/lowercase forms:** `Information` (D1, D2) vs. `information` (D4, D6, D8, D9); `Retrieval` (D2, D10) vs. `retrieval` (D1, D4, D8); `DOCUMENTS` (D9) vs. `documents` (D1–D3, D6–D8, D10); `Search` (D3, D6) vs. `search` (D5, D7).
- **Punctuation:** Periods (`.`) at the end of D1–D10, and a semicolon (`;`) in D7.
- **Repeated terms (within the same document):** `documents` in D2 (`...index documents and documents.`); `the` and `documents` in D7 (`The search engine retrieves documents; the documents are searchable.`).
- **Morphological variants:** `document`/`documents`; `index`/`indexed`/`indexing`; `retrieve`/`retrieves`/`retrieval`; `search`/`searchable`; `system`/`systems`; `engine`/`engines`.

In [7]:
import os
import re
import json
from collections import defaultdict
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords', quiet=True)

corpus = {
    'D1': 'Information retrieval helps users find relevant documents.',
    'D2': 'Information Retrieval systems index documents and documents.',
    'D3': 'Search engines retrieve relevant web documents.',
    'D4': 'Document indexing supports efficient information retrieval.',
    'D5': 'Machine learning improves search engine ranking.',
    'D6': 'Search systems use indexed documents to retrieve information.',
    'D7': 'The search engine retrieves documents; the documents are searchable.',
    'D8': 'Indexing documents makes information retrieval more efficient.',
    'D9': 'DOCUMENTS contain information about document indexing.',
    'D10': 'Retrieval systems organise documents and document collections.'
}

In [8]:
# Part 1: Preprocessing (Versions A, B, C)
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_document(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

tokens_A = {doc_id: preprocess_document(text) for doc_id, text in corpus.items()}
tokens_B = {doc_id: [w for w in tokens if w not in stop_words] for doc_id, tokens in tokens_A.items()}
tokens_C = {doc_id: [stemmer.stem(w) for w in tokens] for doc_id, tokens in tokens_B.items()}

print('Version A:', tokens_A)
print('Version B:', tokens_B)
print('Version C:', tokens_C)

Version A: {'D1': ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents'], 'D2': ['information', 'retrieval', 'systems', 'index', 'documents', 'and', 'documents'], 'D3': ['search', 'engines', 'retrieve', 'relevant', 'web', 'documents'], 'D4': ['document', 'indexing', 'supports', 'efficient', 'information', 'retrieval'], 'D5': ['machine', 'learning', 'improves', 'search', 'engine', 'ranking'], 'D6': ['search', 'systems', 'use', 'indexed', 'documents', 'to', 'retrieve', 'information'], 'D7': ['the', 'search', 'engine', 'retrieves', 'documents', 'the', 'documents', 'are', 'searchable'], 'D8': ['indexing', 'documents', 'makes', 'information', 'retrieval', 'more', 'efficient'], 'D9': ['documents', 'contain', 'information', 'about', 'document', 'indexing'], 'D10': ['retrieval', 'systems', 'organise', 'documents', 'and', 'document', 'collections']}
Version B: {'D1': ['information', 'retrieval', 'helps', 'users', 'find', 'relevant', 'documents'], 'D2': ['information', '

In [9]:
# Part 2: Vocabulary, Binary Matrix, and Term-Frequency Matrix
def build_vocabulary(tokenized_documents):
    vocab = set()
    for tokens in tokenized_documents.values():
        vocab.update(tokens)
    return sorted(vocab)

def build_term_document_matrix(tokenized_documents):
    vocab = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = {term: {doc_id: 1 if term in tokenized_documents[doc_id] else 0 for doc_id in doc_ids} for term in vocab}
    return pd.DataFrame.from_dict(matrix, orient='index', columns=doc_ids)

def build_tf_matrix(tokenized_documents):
    vocab = build_vocabulary(tokenized_documents)
    doc_ids = list(tokenized_documents.keys())
    matrix = {term: {doc_id: tokenized_documents[doc_id].count(term) for doc_id in doc_ids} for term in vocab}
    return pd.DataFrame.from_dict(matrix, orient='index', columns=doc_ids)

display(build_term_document_matrix(tokens_A))
display(build_tf_matrix(tokens_A))

,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,1,1,0,0,1,1,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10
about,0,0,0,0,0,0,0,0,1,0
and,0,1,0,0,0,0,0,0,0,1
are,0,0,0,0,0,0,1,0,0,0
collections,0,0,0,0,0,0,0,0,0,1
contain,0,0,0,0,0,0,0,0,1,0
document,0,0,0,1,0,0,0,0,1,1
documents,1,2,1,0,0,1,2,1,1,1
efficient,0,0,0,1,0,0,0,1,0,0
engine,0,0,0,0,1,0,1,0,0,0
engines,0,0,1,0,0,0,0,0,0,0


### Part 2.5: Comparison of Binary and Term-Frequency Matrices
- **Binary Matrix:** Records only presence (`1`) or absence (`0`) of a term in a document, ignoring repetition (e.g., `documents` in D2 is `1`).
- **Term-Frequency Matrix:** Records the exact count of occurrences of each term in a document (e.g., `documents` in D2 is `2`), capturing term importance/salience within each document.

### Part 3.1: Manual Inverted Index (D1–D3, Version A)
- `information`: `{'D1': [0], 'D2': [0]}`
- `retrieval`: `{'D1': [1], 'D2': [1]}`
- `documents`: `{'D1': [6], 'D2': [4, 6], 'D3': [5]}`

In [10]:
# Part 3 & 4: Positional Inverted Index, Search, and Preprocessing Comparison
def build_inverted_index(tokenized_documents):
    inv_index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in tokenized_documents.items():
        for pos, term in enumerate(tokens):
            inv_index[term][doc_id].append(pos)
    return {term: dict(postings) for term, postings in sorted(inv_index.items())}

def search_term(inverted_index, term):
    return list(inverted_index.get(term, {}).keys())

def compute_metrics(version_name, tokenized_docs, vocab, inv_index):
    vocab_size = len(vocab)
    total_tokens = sum(len(tokens) for tokens in tokenized_docs.values())
    avg_tokens = round(total_tokens / len(tokenized_docs), 2)
    total_postings = sum(len(postings) for postings in inv_index.values())
    total_positions = sum(len(positions) for postings in inv_index.values() for positions in postings.values())
    return {
        'Version': version_name,
        'Vocabulary Size': vocab_size,
        'Avg Tokens/Doc': avg_tokens,
        'Total Postings': total_postings,
        'Total Stored Positions': total_positions
    }

os.makedirs('results', exist_ok=True)
versions = {'A': tokens_A, 'B': tokens_B, 'C': tokens_C}
comparison_rows, indices, bin_matrices, tf_matrices = [], {}, {}, {}

for ver, tok_docs in versions.items():
    vocab = build_vocabulary(tok_docs)
    bin_df = build_term_document_matrix(tok_docs)
    tf_df = build_tf_matrix(tok_docs)
    inv_idx = build_inverted_index(tok_docs)
    bin_matrices[ver], tf_matrices[ver], indices[ver] = bin_df, tf_df, inv_idx
    with open(f'results/tokens_{ver}.json', 'w') as f: json.dump(tok_docs, f, indent=2)
    with open(f'results/vocabulary_{ver}.txt', 'w') as f: f.write('\n'.join(vocab))
    bin_df.to_csv(f'results/binary_matrix_{ver}.csv')
    tf_df.to_csv(f'results/tf_matrix_{ver}.csv')
    with open(f'results/inverted_index_{ver}.json', 'w') as f: json.dump(inv_idx, f, indent=2)
    comparison_rows.append(compute_metrics(f'Version {ver}', tok_docs, vocab, inv_idx))

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv('results/preprocessing_comparison.csv', index=False)
display(comparison_df)

,Version,Vocabulary Size,Avg Tokens/Doc,Total Postings,Total Stored Positions
0,Version A,36,6.9,66,69
1,Version B,30,6.1,59,61
2,Version C,24,6.1,57,61


In [11]:
# Part 5: Verification Tests
assert search_term(indices['B'], 'machine') == ['D5']
assert search_term(indices['B'], 'information') == ['D1', 'D2', 'D4', 'D6', 'D8', 'D9']
assert search_term(indices['B'], 'neural') == []
assert search_term(indices['B'], 'retrieves') == ['D7']
assert search_term(indices['C'], 'retrieves') == []
assert search_term(indices['C'], 'retriev') == ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D10']
assert indices['A']['documents']['D2'] == [4, 6]
assert indices['A']['retrieval']['D1'] == [1]
assert set(bin_matrices['A'].values.flatten()).issubset({0, 1})
assert tf_matrices['A'].loc['documents', 'D2'] == 2
print('All verification checks passed!')

All verification checks passed!
